In [1]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Input, Conv1D, Flatten, Dense, Permute
from tensorflow.keras.regularizers import l2

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from numpy.lib.stride_tricks import as_strided

I0000 00:00:1788831574.455607   35917 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Loading CSV

In [5]:
DATASET_DIR = "../dataset/phase1"
CMS_TAG = "cms1"
SEQ_LEN = 816
DIVISOR = 200000.0
NUM_CLASSES = 2
CLASSES = np.array(['benign', 'malware'])

def load_data():
    bdir = os.path.join(DATASET_DIR, f'benign_{CMS_TAG}')
    mdir = os.path.join(DATASET_DIR, f'malware_{CMS_TAG}')
    bfiles = sorted(glob.glob(os.path.join(bdir, 'cms_*.csv')))
    mfiles = sorted(glob.glob(os.path.join(mdir, 'cms_*.csv')))

    def load_all(files):
        out = []
        for f in files:
            v = pd.read_csv(f, header=None).values.flatten().astype(np.float32)
            out.append(v)
        return np.array(out, dtype=np.float32)

    Xb = load_all(bfiles)
    Xm = load_all(mfiles)
    X = np.concatenate([Xb, Xm], axis=0)
    y = np.concatenate([np.zeros(len(Xb), dtype=np.int64), np.ones(len(Xm), dtype=np.int64)])
    return X, y

In [6]:
X, y = load_data()

## Train, Validation, Test Split and Normalize

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = np.clip(X_train / DIVISOR, 0.0, 1.0)
X_val = np.clip(X_val / DIVISOR, 0.0, 1.0)
X_test = np.clip(X_test / DIVISOR, 0.0, 1.0)

X_train = X_train.reshape(-1, SEQ_LEN, 1)
X_val = X_val.reshape(-1, SEQ_LEN, 1)
X_test = X_test.reshape(-1, SEQ_LEN, 1)

## 1D CNN model

In [8]:
input_layer = Input(shape=(SEQ_LEN, 1))

x = Conv1D(filters=16, kernel_size=3, strides=10, padding='valid', activation='relu')(input_layer)
x = Conv1D(filters=32, kernel_size=3, strides=1, padding='valid', activation='relu')(x)
x = Conv1D(filters=64, kernel_size=3, strides=1, padding='valid', activation='relu')(x)

x = Permute((2, 1))(x)
x = Flatten()(x)

x = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(x)
output_layer = Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=l2(1e-4))(x)

model = Model(input_layer, output_layer)

opt = Adam(learning_rate=0.001)
model.compile(loss='sparse_categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

## Check Point

In [11]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=0.00001)
checkpoint = ModelCheckpoint(
    filepath='./phase1_cms1.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

## Model Training

In [12]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), callbacks=[reduce_lr, checkpoint])

Epoch 1/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8769 - loss: 0.2776
Epoch 1: val_accuracy improved from None to 0.88099, saving model to ./phase1_cms1.h5



Epoch 1: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8752 - loss: 0.2784 - val_accuracy: 0.8810 - val_loss: 0.2282 - learning_rate: 0.0010
Epoch 2/100
66/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9129 - loss: 0.2180
Epoch 2: val_accuracy improved from 0.88099 to 0.93961, saving model to ./phase1_cms1.h5



Epoch 2: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9125 - loss: 0.2178 - val_accuracy: 0.9396 - val_loss: 0.1697 - learning_rate: 0.0010
Epoch 3/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9317 - loss: 0.1791
Epoch 3: val_accuracy improved from 0.93961 to 0.94849, saving model to ./phase1_cms1.h5



Epoch 3: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9316 - loss: 0.1815 - val_accuracy: 0.9485 - val_loss: 0.1621 - learning_rate: 0.0010
Epoch 4/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9351 - loss: 0.1811
Epoch 4: val_accuracy did not improve from 0.94849
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9351 - loss: 0.1811 - val_accuracy: 0.9290 - val_loss: 0.1974 - learning_rate: 0.0010
Epoch 5/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9411 - loss: 0.1697
Epoch 5: val_accuracy did not improve from 0.94849
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9409 - loss: 0.1693 - val_accuracy: 0.9485 - val_loss: 0.1530 - learning_rate: 0.0010
Epoch 6/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9433 - loss: 0.1777
Epoch 6: val_accuracy did not improve from 0.94849
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9445 - loss: 0.1722 - val_accuracy: 0.9325 - val_loss: 0.1657 -


Epoch 9: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9378 - loss: 0.1660 - val_accuracy: 0.9503 - val_loss: 0.1452 - learning_rate: 0.0010
Epoch 10/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9429 - loss: 0.1553
Epoch 10: val_accuracy did not improve from 0.95027
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9436 - loss: 0.1542 - val_accuracy: 0.9485 - val_loss: 0.1472 - learning_rate: 0.0010
Epoch 11/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9407 - loss: 0.1637
Epoch 11: val_accuracy did not improve from 0.95027
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9414 - loss: 0.1622 - val_accuracy: 0.9449 - val_loss: 0.1501 - learning_rate: 0.0010
Epoch 12/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9481 - loss: 0.1545
Epoch 12: val_accuracy did not improve from 0.95027
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9494 - loss: 0.1525 - val_accuracy: 0.9503 - val_loss: 0.1

...


Epoch 39: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9591 - loss: 0.1279 - val_accuracy: 0.9645 - val_loss: 0.1140 - learning_rate: 5.0000e-04
Epoch 40/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9592 - loss: 0.1286
Epoch 40: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9596 - loss: 0.1277 - val_accuracy: 0.9645 - val_loss: 0.1149 - learning_rate: 5.0000e-04
Epoch 41/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9577 - loss: 0.1297
Epoch 41: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9578 - loss: 0.1287 - val_accuracy: 0.9627 - val_loss: 0.1134 - learning_rate: 5.0000e-04
Epoch 42/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9579 - loss: 0.1308
Epoch 42: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9578 - loss: 0.1307 - val_accuracy: 0.9627 -


Epoch 44: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9582 - loss: 0.1267 - val_accuracy: 0.9663 - val_loss: 0.1139 - learning_rate: 5.0000e-04
Epoch 45/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9616 - loss: 0.1207
Epoch 45: val_accuracy did not improve from 0.96625
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9605 - loss: 0.1218 - val_accuracy: 0.9609 - val_loss: 0.1144 - learning_rate: 5.0000e-04
Epoch 46/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9563 - loss: 0.1265
Epoch 46: val_accuracy did not improve from 0.96625
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9565 - loss: 0.1264 - val_accuracy: 0.9556 - val_loss: 0.1259 - learning_rate: 5.0000e-04
Epoch 47/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9598 - loss: 0.1262
Epoch 47: val_accuracy did not improve from 0.96625
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9600 - loss: 0.1259 - val_accuracy: 0.9663 -


Epoch 56: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9587 - loss: 0.1226 - val_accuracy: 0.9680 - val_loss: 0.1085 - learning_rate: 5.0000e-04
Epoch 57/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9600 - loss: 0.1194
Epoch 57: val_accuracy did not improve from 0.96803
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9596 - loss: 0.1229 - val_accuracy: 0.9645 - val_loss: 0.1088 - learning_rate: 5.0000e-04
Epoch 58/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9609 - loss: 0.1200
Epoch 58: val_accuracy did not improve from 0.96803
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9609 - loss: 0.1200 - val_accuracy: 0.9645 - val_loss: 0.1130 - learning_rate: 5.0000e-04
Epoch 59/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9582 - loss: 0.1253
Epoch 59: val_accuracy did not improve from 0.96803
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9582 - loss: 0.1248 - val_accuracy: 0.9645 - v


Epoch 75: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9618 - loss: 0.1166 - val_accuracy: 0.9698 - val_loss: 0.1030 - learning_rate: 2.5000e-04
Epoch 76/100
64/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9614 - loss: 0.1204
Epoch 76: val_accuracy did not improve from 0.96980
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9609 - loss: 0.1199 - val_accuracy: 0.9645 - val_loss: 0.1095 - learning_rate: 2.5000e-04
Epoch 77/100
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9615 - loss: 0.1184
Epoch 77: val_accuracy did not improve from 0.96980
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9627 - loss: 0.1150 - val_accuracy: 0.9663 - val_loss: 0.1097 - learning_rate: 2.5000e-04
Epoch 78/100
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9636 - loss: 0.1155
Epoch 78: val_accuracy did not improve from 0.96980
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9636 - loss: 0.1157 - val_accuracy: 0.9663 - v


Epoch 87: finished saving model to ./phase1_cms1.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9614 - loss: 0.1134 - val_accuracy: 0.9716 - val_loss: 0.1018 - learning_rate: 2.5000e-04
Epoch 88/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9616 - loss: 0.1112
Epoch 88: val_accuracy did not improve from 0.97158
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9614 - loss: 0.1117 - val_accuracy: 0.9716 - val_loss: 0.1014 - learning_rate: 2.5000e-04
Epoch 89/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9629 - loss: 0.1123
Epoch 89: val_accuracy did not improve from 0.97158
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9631 - loss: 0.1118 - val_accuracy: 0.9680 - val_loss: 0.1050 - learning_rate: 2.5000e-04
Epoch 90/100
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9619 - loss: 0.1133
Epoch 90: val_accuracy did not improve from 0.97158
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9614 - loss: 0.1142 - val_accuracy: 0.9680 - v

## Evaluate (float32)

In [13]:
cp_model = load_model('./phase1_cms1.h5')
cp_model.evaluate(X_test, y_test, batch_size=1000)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.9635 - loss: 0.1146 


[0.11456257104873657, 0.9635157585144043]

In [14]:
y_pred = cp_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes, target_names=list(CLASSES), digits=4))

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
              precision    recall  f1-score   support

      benign     0.9573    0.9727    0.9650       623
     malware     0.9703    0.9537    0.9619       583

    accuracy                         0.9635      1206
   macro avg     0.9638    0.9632    0.9635      1206
weighted avg     0.9636    0.9635    0.9635      1206



In [15]:
conf_matrix = confusion_matrix(y_test, y_pred_classes)
conf_matrix_df = pd.DataFrame(conf_matrix, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix:")
print(conf_matrix_df)

Confusion Matrix:
         benign  malware
benign      606       17
malware      27      556


## Q15 Quantization

In [16]:
def scale_for_q15(w):
    absmax = np.abs(w).max()
    if absmax <= 1.0:
        return w, 1
    scale = 1
    while absmax / scale > 1.0:
        scale *= 2
    return w / scale, scale

def to_q15(x):
    return np.clip(np.round(x * 32768.0), -32768, 32767).astype(np.int64)

def to_q15_bias(x):
    return np.clip(np.round(x * 32768.0), -2**31, 2**31 - 1).astype(np.int64)

def quantize_weights_flatten(model):
    conv_layers = [l for l in model.layers if 'conv1d' in l.name]
    dense_layers = [l for l in model.layers if l.name.startswith('dense')]
    c1_w_f, c1_b_f = conv_layers[0].get_weights()
    c2_w_f, c2_b_f = conv_layers[1].get_weights()
    c3_w_f, c3_b_f = conv_layers[2].get_weights()
    d_w_f, d_b_f = dense_layers[0].get_weights()
    fc_w_f, fc_b_f = dense_layers[1].get_weights()
    c1_w = np.transpose(c1_w_f[:, 0, :], (1, 0))
    c2_w = np.transpose(c2_w_f, (2, 1, 0))
    c3_w = np.transpose(c3_w_f, (2, 1, 0))
    d_w = d_w_f.T
    fc_w = fc_w_f.T
    out = {}
    for name, w, b in [('conv1', c1_w, c1_b_f), ('conv2', c2_w, c2_b_f), ('conv3', c3_w, c3_b_f),
                        ('dense', d_w, d_b_f), ('fc2', fc_w, fc_b_f)]:
        w_s, scale = scale_for_q15(w)
        out[f'{name}_w'] = to_q15(w_s)
        out[f'{name}_b'] = to_q15_bias(b / scale)
    return out

def windows_1d(a, out_len, k, stride, axis):
    a = np.ascontiguousarray(a)
    shape = list(a.shape); shape[axis] = out_len; shape = shape + [k]
    strides = list(a.strides); ts = strides[axis]
    strides[axis] = ts * stride; strides = strides + [ts]
    return as_strided(a, shape=shape, strides=strides)

def q15_forward(qw, X, s1, s2, s3, seq_len):
    c1w, c1b = qw['conv1_w'].astype(np.int64), qw['conv1_b'].astype(np.int64)
    c2w, c2b = qw['conv2_w'].astype(np.int64), qw['conv2_b'].astype(np.int64)
    c3w, c3b = qw['conv3_w'].astype(np.int64), qw['conv3_b'].astype(np.int64)
    dw, db = qw['dense_w'].astype(np.int64), qw['dense_b'].astype(np.int64)
    fcw, fcb = qw['fc2_w'].astype(np.int64), qw['fc2_b'].astype(np.int64)
    F1, K1 = c1w.shape
    F2, _, K2 = c2w.shape
    F3, _, K3 = c3w.shape
    c1out = (seq_len - K1) // s1 + 1
    c2out = (c1out - K2) // s2 + 1
    c3out = (c2out - K3) // s3 + 1
    Xq = np.clip(np.round(X * 32768), -32768, 32767).astype(np.int64)
    win1 = windows_1d(Xq, c1out, K1, s1, axis=1)
    a1 = np.maximum(0, (np.einsum('ntk,fk->nft', win1, c1w) >> 15) + c1b[None, :, None])
    a1c = np.clip(a1, -32768, 32767)
    win2 = windows_1d(a1c, c2out, K2, s2, axis=2)
    a2 = np.maximum(0, (np.einsum('nctk,fck->nft', win2, c2w) >> 15) + c2b[None, :, None])
    a2c = np.clip(a2, -32768, 32767)
    win3 = windows_1d(a2c, c3out, K3, s3, axis=2)
    a3 = np.maximum(0, (np.einsum('nctk,fck->nft', win3, c3w) >> 15) + c3b[None, :, None])
    a3c = np.clip(a3, -32768, 32767)
    N = X.shape[0]
    flat = a3c.reshape(N, F3 * c3out)
    hidden = np.clip(np.maximum(0, (flat @ dw.T >> 15) + db[None, :]), -32768, 32767)
    logits = (hidden @ fcw.T >> 15) + fcb[None, :]
    return np.argmax(logits, axis=1)

## Evaluate (Q15)

In [17]:
qw = quantize_weights_flatten(cp_model)
X_test_flat = X_test.reshape(-1, SEQ_LEN)
y_pred_q15 = q15_forward(qw, X_test_flat, 10, 1, 1, SEQ_LEN)

print(classification_report(y_test, y_pred_q15, target_names=list(CLASSES), digits=4))

              precision    recall  f1-score   support

      benign     0.9575    0.9759    0.9666       623
     malware     0.9737    0.9537    0.9636       583

    accuracy                         0.9652      1206
   macro avg     0.9656    0.9648    0.9651      1206
weighted avg     0.9653    0.9652    0.9652      1206



In [18]:
conf_matrix_q15 = confusion_matrix(y_test, y_pred_q15)
conf_matrix_q15_df = pd.DataFrame(conf_matrix_q15, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix (Q15):")
print(conf_matrix_q15_df)

Confusion Matrix (Q15):
         benign  malware
benign      608       15
malware      27      556
